# Lab 6 - Mini Project : Predictive Maintenance in High-Performance Engineering
**Course:** NoSQL Database & Data Analysis
**Domain:** Automotive & Mechanical Engineering
**Objective:** Apply a complete data analysis workflow to predict mechanical failures based on telemetry and sensor data (RPM, Torque, Temperatures).

In [ ]:
# Importing required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Setting plotting style
plt.style.use('ggplot')

## 1. Dataset Selection and Description
This project utilizes the **AI4I 2020 Predictive Maintenance** dataset (Kaggle). It simulates the operational parameters of industrial and mechanical equipment.
- **Source:** Kaggle (Synthetic data based on real-world engineering cases).
- **Key Features:** Rotational speed (RPM), Torque (Nm), Temperatures (Air & Process), Tool wear.
- **Target:** `Machine failure` (0 = Normal operation, 1 = Mechanical failure).

In [ ]:
# Loading the dataset
df = pd.read_csv('predictive_maintenance.csv')

# Initial overview
print("Dataset dimensions:", df.shape)

# --- TP2 : Data Cleaning & Data Quality ---
# Checking for missing values
missing_values = df.isnull().sum().sum()
print(f"Missing values detected: {missing_values}")

# If there were missing values, this is how we would apply the mean (TP2 Recap):
# df.fillna(df.mean(numeric_only=True), inplace=True)

# Dropping irrelevant columns for modeling (Identifiers)
df = df.drop(['UDI', 'Product ID'], axis=1)
df.head()

## 2. Exploratory Data Analysis (EDA) - (TP3 Reference)
As an engineer, the goal of this phase is to understand the physical correlations between the sensors. For instance, how does temperature evolve under specific torque and speed loads?

In [ ]:
# Descriptive statistics
display(df.describe())

# Correlation matrix to identify key relationships (e.g., Temperature vs. Tool Wear)
plt.figure(figsize=(10, 8))
corr_matrix = df.select_dtypes(include=[np.number]).corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Mechanical Sensors Correlation Matrix")
plt.show()

# Scatter Plot (TP3 Recap: Temperature vs Torque/Vibration)
# Here, we observe the relationship between Torque and Rotational Speed (RPM)
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x='Rotational speed [rpm]', y='Torque [Nm]', hue='Machine failure', palette=['blue', 'red'], alpha=0.6)
plt.title("Relationship between Torque and RPM (Failure Zone Detection)")
plt.xlabel("Rotational Speed (RPM)")
plt.ylabel("Torque (Nm)")
plt.show()

### Continuous Monitoring Simulation (TP5 - Time Series)
Although this dataset lacks an explicit timestamp, we can simulate the evolution of the process temperature over time (using the index) and apply a rolling average to smooth out sensor noise.

In [ ]:
# Applying a rolling average over a 50-measurement window (TP5 Recap)
plt.figure(figsize=(12, 5))
plt.plot(df['Process temperature [K]'].iloc[:500], label='Raw Temperature', alpha=0.5)
plt.plot(df['Process temperature [K]'].iloc[:500].rolling(window=20).mean(), color='red', label='Rolling Average (Smoothed)', linewidth=2)
plt.title("Temperature Evolution Over Time (Streaming Simulation)")
plt.xlabel("Measurement Index (Time)")
plt.ylabel("Temperature (K)")
plt.legend()
plt.show()

## 3. Preprocessing and Feature Engineering
Preparing the data for the machine learning algorithm:
- **Encoding:** Converting the categorical variable 'Type' (Quality L, M, H) into numeric values.
- **Scaling:** Standardizing the data is mandatory for the KNN algorithm and PCA analysis to prevent features with larger scales (like RPM) from dominating the model.

In [ ]:
# Separating features (X) and target (y)
# We exclude specific failure type columns to predict the overall failure event
X = df.drop(['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNQ'], axis=1)
y = df['Machine failure']

# Encoding the 'Type' variable
le = LabelEncoder()
X['Type'] = le.fit_transform(X['Type'])

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Feature Scaling (StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 4. Modeling and Analysis
We are utilizing two techniques covered in the labs:
1. **PCA (Principal Component Analysis):** To reduce dimensionality and visualize the separation between normal operational data and mechanical failures.
2. **KNN (K-Nearest Neighbors):** A classification algorithm to predict mechanical failure based on the proximity to other known operational states.

In [ ]:
# --- PCA Analysis ---
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_train_scaled)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_train, cmap='bwr', alpha=0.5)
plt.title("PCA Analysis: 2D Projection of System States")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(handles=scatter.legend_elements()[0], labels=['Normal', 'Failure'])
plt.show()

# --- KNN Modeling ---
# Initializing and training the model
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

# Making predictions
y_pred = knn.predict(X_test_scaled)

## 5. Results and Interpretation

In [ ]:
# Displaying the confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted: Normal', 'Predicted: Failure'],
            yticklabels=['Actual: Normal', 'Actual: Failure'])
plt.title("Confusion Matrix - KNN Model")
plt.show()

# Classification Report
print("Classification Report:\n")
print(classification_report(y_test, y_pred))

## 6. Conclusion and Perspectives
**Analysis Summary:**
The KNN model, combined with rigorous data scaling (TP2), proves capable of identifying conditions that lead to mechanical failure. The exploratory data analysis (TP3) highlighted that failures often cluster at the extremes of the power curve (e.g., high torque at low RPMs or severe thermal overload).

**Engineering Perspectives & Improvements:**
- **Class Imbalance:** Because failures are rare (typical in reliability engineering), using techniques like SMOTE could generate synthetic failure data to better train the algorithm.
- **Real-World Deployment:** In a motorsport or performance tuning scenario, this type of predictive model could be integrated directly into an ECU as an active safety net (e.g., automatically cutting ignition or reducing boost pressure the millisecond telemetry enters a failure cluster identified by the KNN).